[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_03_order_parameters_slaving.ipynb)


# 03. Order parameters and the slaving principle

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** When may fast variables be expressed as functions of a few slow unstable modes?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- identify soft and stable modes from their relaxation rates
- derive a leading-order slaving relation
- quantify the error and breakdown of adiabatic elimination


## Prerequisite bridge: time-scale separation

Suppose $A$ changes on time $T_s$ and $B$ on a much shorter time $T_f$. After scaling by $T_s$, the fast equation carries $\varepsilon=T_f/T_s\ll1$. During an initial layer of duration $O(\varepsilon)$, $B$ rapidly approaches a slow manifold. Thereafter it can often be approximated by $B=h(A)$.

Setting the fast derivative to zero is the leading approximation, not an identity. It must be tested by varying $\varepsilon$, excluding the initial layer, and checking that the fast equilibrium remains stable along the trajectory. Near a second instability, resonance, or loss of normal hyperbolicity, slaving can fail.


## Fast–slow prototype

Consider

$$\dot A=\mu A-A^3+cAB,\qquad \varepsilon\dot B=-B+dA^2.$$

Near $\mu=0$, $A$ is slow while $B$ relaxes at rate $1/\varepsilon$. Setting the fast derivative to zero gives the slaving relation $B\approx dA^2$ and the reduced amplitude equation

$$\dot A=\mu A-(1-cd)A^3.$$

This is not merely deletion of a variable: it is an approximation whose error depends on scale separation, distance from instability, initial layers, and normal hyperbolicity of the fast manifold.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

from scipy.integrate import solve_ivp

mu, c, d = 0.15, 0.35, 0.8
t_eval = np.linspace(0, 80, 3000)

def full_system(t, y, eps):
    A, B = y
    return mu*A - A**3 + c*A*B, (-B + d*A**2)/eps

def reduced_system(t, y):
    A = y[0]
    return [mu*A - (1-c*d)*A**3]

reduced = solve_ivp(reduced_system, (0, 80), [0.08], t_eval=t_eval, rtol=1e-9, atol=1e-11)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
errors = []
for eps in [0.5, 0.2, 0.05]:
    full = solve_ivp(full_system, (0, 80), [0.08, -0.4], args=(eps,), t_eval=t_eval,
                     rtol=1e-9, atol=1e-11)
    ax[0].plot(t_eval, full.y[0], label=fr"full $\epsilon={eps}$")
    mask = t_eval > 5*eps
    errors.append(np.max(np.abs(full.y[0, mask] - reduced.y[0, mask])))
ax[0].plot(t_eval, reduced.y[0], "k--", lw=2, label="reduced")
ax[0].set(xlabel="time", ylabel="A", title="Full and reduced dynamics")
ax[0].legend()
ax[1].loglog([0.5, 0.2, 0.05], errors, "o-")
ax[1].set(xlabel=r"$\epsilon$", ylabel="max post-layer error", title="Reduction error")
plt.tight_layout()


## Center-manifold viewpoint

The graph $B=h(A,\mu)$ must satisfy an invariance equation, not just $\dot B=0$. Expanding $h=dA^2+O(\varepsilon, A^4,\mu A^2)$ supplies systematic corrections. The slaving principle is therefore a physical interpretation of a local invariant-manifold reduction near a loss of stability.

> **Checkpoint:** Fast relaxation alone is insufficient if the fast subsystem loses stability, if forcing resonates with it, or if the trajectory leaves the neighborhood where the expansion is valid.


## Assessed exercise

Derive the first $O(\varepsilon)$ correction to $B=h(A)$ from the invariance equation. Add it to the reduced model, repeat the error plot, and determine the observed convergence order in $\varepsilon$.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** Haken, *Synergetics: An Introduction* and *Synergetics: Introduction and Advanced Topics*, order parameters and slaving principle.

**Verified web reading:**
- [Haken's Synergetics: An Introduction](https://doi.org/10.1007/978-3-642-88338-5)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
